## Proyecto Integrador

Requisitos:
1. Genera datos aleatorios de calificaciones para 50 estudiantes  
2. Calcular promedios individuales y por materia  
3. Determinar aprobados y reprobados  
4. Generar al menos 4 visualizaciones diferentes  
5. Presentar un resumen estadístico

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Clases

In [ ]:
class Estudiante:
  '''Clase que representa un estudiante'''

  def __init__(self, id_estudiante: int):
    '''Constructor de la clase Estudiante'''
    self.id = id_estudiante
    self.nombre = f'Estudiante_{id_estudiante:02d}'
    self.calificaciones = {}
    self.promedio = 0.0
    self.estado = 'Sin evaluar'

  def agregar_calificacion(self, materia: str, calificacion: float):
    '''Método para agregar calificación'''
    self.calificaciones[materia] = calificacion

  def calcular_promedio(self) -> float:
    '''Método para calcular promedio individual'''
    if len(self.calificaciones) > 0:
      califs = list(self.calificaciones.values())
      self.promedio = np.mean(califs)
    return self.promedio

  def determinar_estado(self) -> str:
    '''Método para determinar si aprueba, va a extraordinario o reprueba'''
    if len(self.calificaciones) == 0:
      self.estado = 'Sin calificaciones'
      return self.estado

    aprobadas_count = sum(1 for calif in self.calificaciones.values() if calif >= 60)
    total_materias = len(self.calificaciones)

    if aprobadas_count == total_materias:
      self.estado = 'APROBADO'
    elif aprobadas_count == 3:
      self.estado = 'EXTRAORDINARIO'
    else:
      self.estado = 'REPROBADO'
    return self.estado

  def obtener_info(self) -> dict:
    '''Método para obtener información del estudiante'''
    return {
            'id': self.id,
            'nombre': self.nombre,
            'promedio': round(self.promedio, 1),
            'estado': self.estado,
            'calificaciones': self.calificaciones
           }

  def __str__(self) -> str:
    '''Método especial para mostrar como enunciado'''
    return f'{self.nombre}: Promedio={self.promedio:.1f}, Estado={self.estado}'


class Materia:
  '''Clase que representa una materia académica'''

  def __init__(self, nombre: str):
    '''Constructor de la clase Materia'''
    self.nombre = nombre
    self.calificaciones = []
    self.promedio = 0.0
    self.aprobados = 0
    self.reprobados = 0

  def agregar_calificaciones(self, calificaciones: list):
    '''Método para agregar calificaciones'''
    self.calificaciones = calificaciones
    self._calcular_estadisticas()

  def _calcular_estadisticas(self):
    '''Método privado para calcular estadísticas'''
    if len(self.calificaciones) > 0:
      self.promedio = np.mean(self.calificaciones)
      self.minimo = np.min(self.calificaciones)
      self.maximo = np.max(self.calificaciones)
      self.mediana = np.median(self.calificaciones)

      self.aprobados = np.sum(np.array(self.calificaciones) >= 60)
      self.reprobados = len(self.calificaciones) - self.aprobados

  def obtener_estadisticas(self) -> dict:
    '''Método para obtener estadísticas'''
    return {
            'nombre': self.nombre,
            'promedio': round(self.promedio, 1),
            'minimo': round(self.minimo, 1),
            'maximo': round(self.maximo, 1),
            'mediana': round(self.mediana, 1),
            'aprobados': self.aprobados,
            'reprobados': self.reprobados,
            'porcentaje_aprobados': round((self.aprobados / len(self.calificaciones)) * 100, 1)
            }

  def __str__(self) -> str:
    '''Método especial para mostrar como enunciado'''
    return f'Materia: {self.nombre}, Promedio: {self.promedio:.1f}'

# Funciones

In [ ]:
def generar_datos_estudiantes():
  '''Función para generar datos aleatorios'''
  np.random.seed(46)

  materias = ['Cálculo', 'Biología', 'Filosofía', 'Etimología']
  n_estudiantes = 50

  objetos_materias = [] # Crea objetos Materias
  for materia_nombre in materias:
    materia = Materia(materia_nombre)

    if materia_nombre == 'Cálculo':     # Genera calificaciones con diferentes distribuciones para cada materia
      califs = np.clip(np.random.normal(65, 15, n_estudiantes), 0, 100)
    elif materia_nombre == 'Biología':
      califs = np.clip(np.random.normal(75, 10, n_estudiantes), 0, 100)
    elif materia_nombre == 'Filosofía':
      califs = np.random.uniform(40, 90, n_estudiantes)
    else:  # Etimología
      califs = np.clip(np.random.normal(70, 20, n_estudiantes), 0, 100)

    materia.agregar_calificaciones(np.round(califs, 1).tolist())
    objetos_materias.append(materia)

  estudiantes = []  # Crea objetos Estudiante
  for i in range(n_estudiantes):
    estudiante = Estudiante(i + 1)

    for materia_obj in objetos_materias:
      calificacion = materia_obj.calificaciones[i]
      estudiante.agregar_calificacion(materia_obj.nombre, calificacion)

    estudiante.calcular_promedio()
    estudiante.determinar_estado()
    estudiantes.append(estudiante)

  return estudiantes, objetos_materias


def calcular_estadisticas_generales(estudiantes, materias):
  '''Función para calcular estadísticas generales'''
  promedios = [e.promedio for e in estudiantes]
  aprobados_total = sum(1 for e in estudiantes if e.estado == 'APROBADO')
  extraordinarios_total = sum(1 for e in estudiantes if e.estado == 'EXTRAORDINARIO')

  top_5 = sorted(estudiantes, key=lambda x: x.promedio, reverse=True)[:5]

  return {
        'total_estudiantes': len(estudiantes),
        'promedio_general': round(np.mean(promedios), 1),
        'minimo_general': round(np.min(promedios), 1),
        'maximo_general': round(np.max(promedios), 1),
        'aprobados_total': aprobados_total,
        'porcentaje_aprobados': round((aprobados_total / len(estudiantes)) * 100, 1),
        'extraordinarios_total': extraordinarios_total,
        'porcentaje_extraordinarios': round((extraordinarios_total / len(estudiantes)) * 100, 1),
        'top_5_estudiantes': top_5,
        'estadisticas_materias': [m.obtener_estadisticas() for m in materias]
        }


def crear_graficas(estudiantes, materias):
  '''Función para crear las gráficas'''
  nombres_materias = [m.nombre for m in materias]
  promedios_materias = [m.promedio for m in materias]
  porcentaje_aprobacion_materia = [(m.aprobados / len(estudiantes)) * 100 for m in materias]
  promedios_individuales = [e.promedio for e in estudiantes]

  plt.figure(figsize=(14,8))

  # 1. Barras - Promedios por materia
  plt.subplot(2,2,1)
  plt.bar(nombres_materias, promedios_materias, color=['blue', 'green', 'red', 'purple'])
  plt.axhline(y=60, color='maroon', linestyle='--', label='Calificación Aprobatoria (60)')
  plt.title('Promedio por Materia', fontsize=12, fontweight='bold')
  plt.xlabel('Materias')
  plt.ylabel('Calificación Promedio')
  plt.grid(True, alpha=0.3)
  plt.legend()

  # 2. Histograma - Distribución de promedios individuales
  plt.subplot(2,2,2)
  plt.hist(promedios_individuales, bins=10, color='blue', edgecolor='black')
  plt.axvline(x=60, color='maroon', linestyle='--', label='Calificación Aprobatoria (60)')
  plt.title('Distribución de Promedios Individuales', fontsize=12, fontweight='bold')
  plt.xlabel('Promedio')
  plt.ylabel('Número de Estudiantes')
  plt.grid(True, alpha=0.3)
  plt.legend()


  # 3. Pastel - Distribución de Estudiantes por Estado
  plt.subplot(2,2,3)
  estados = [e.estado for e in estudiantes]
  conteo_estados = {estado: estados.count(estado) for estado in set(estados)}
  etiquetas = list(conteo_estados.keys())
  tamanios = list(conteo_estados.values())
  plt.pie(tamanios, labels=etiquetas, autopct='%1.1f%%', colors=['orange', 'red', 'green'], startangle=90)
  plt.title('Distribución de Estudiantes por Estado Aprobatorio', fontsize=12, fontweight='bold')


  # 4. Dispersión - Relación entre promedio general y materias por estado
  plt.subplot(2,2,4)
  califs_total = promedios_individuales
  califs_materia1 = materias[0].calificaciones
  plt.scatter(califs_total, califs_materia1, alpha=0.6, c='blue')
  plt.title(f'Relación entre el Promedio General y la Materia de {materias[0].nombre}', fontsize=12, fontweight='bold')
  plt.xlabel(f'Promedio General Individual')
  plt.ylabel(f'Calificaciones de {materias[0].nombre}')
  plt.grid(True, alpha=0.3)
  plt.legend()
  # Línea de dispersión
  coef = np.polyfit(califs_total, califs_materia1,1)
  poly1d_fn = np.poly1d(coef)
  plt.plot(califs_total, poly1d_fn(califs_total), 'maroon', linewidth=2, label = 'línea de tendencia')

  plt.tight_layout()
  plt.show()


def mostrar_resumen_estadistico(estadisticas):
  '''Función para mostrar resumen estadístico'''
  print('=' * 80)
  print('REPORTE DEL PROYECTO')
  print('=' * 80)

  print(f'\nDATOS GENERALES:')
  print(f'• Total de estudiantes: {estadisticas['total_estudiantes']}')
  print(f'• Promedio general: {estadisticas['promedio_general']}')
  print(f'• Rango de promedios: {estadisticas['minimo_general']} - {estadisticas['maximo_general']}')
  print(f'• Estudiantes aprobados: {estadisticas['aprobados_total']} ({estadisticas['porcentaje_aprobados']}%)')
  print(f'• Estudiantes en proceso extraordinario: {estadisticas['extraordinarios_total']} ({estadisticas['porcentaje_extraordinarios']}%)')

  print(f'\nESTADÍSTICAS POR MATERIA:')
  print('-' * 80)
  print(f'{'Materia':<12} {'Promedio':<8} {'Mín':<6} {'Máx':<6} {'Aprobados':<10} {'%Aprob':<8}')
  print('-' * 80)

  for stats in estadisticas['estadisticas_materias']:
    print(f'{stats['nombre']:<12} {stats['promedio']:<8.1f} {stats['minimo']:<6.1f} {stats['maximo']:<6.1f} {stats['aprobados']:<10} {stats['porcentaje_aprobados']:<8.1f}')

  print(f'\nTOP 5 ESTUDIANTES:')
  print('-' * 80)
  for i, estudiante in enumerate(estadisticas['top_5_estudiantes'], 1):
    califs_str = ' | '.join([f'{m[:3]}:{c}' for m, c in estudiante.calificaciones.items()])
    print(f'{i}. {estudiante.nombre} - Promedio: {estudiante.promedio:.1f} - {califs_str}')

  # Encontrar materia más difícil y más fácil
  materias_stats = estadisticas['estadisticas_materias']
  materia_mas_dificil = min(materias_stats, key=lambda x: x['promedio'])
  materia_mas_facil = max(materias_stats, key=lambda x: x['promedio'])

  print(f'\nANÁLISIS DE DIFICULTAD:')
  print('-' * 80)
  print(f'• Materia con menos aprobados: {materia_mas_dificil['nombre']} (Promedio: {materia_mas_dificil['promedio']})')
  print(f'• Materia con más aprobados: {materia_mas_facil['nombre']} (Promedio: {materia_mas_facil['promedio']})')
  print('=' * 80)


def mostrar_ejemplo_estudiantes(estudiantes, n=3):
  '''Función para mostrar ejemplo de algunos estudiantes'''
  print(f'\nEJEMPLO DE {n} ESTUDIANTES:')
  print('-' * 80)

  for i in range(min(n, len(estudiantes))):
    estudiante = estudiantes[i]
    print(f'\n{estudiante.nombre}:')
    print(f'  Promedio: {estudiante.promedio:.1f}')

    print(f'  Calificaciones:')
    for materia, calif in estudiante.calificaciones.items():
      estado = 'APROBADO' if calif >= 60 else 'REPROBADO'
      print(f'    • {materia}: {calif:.1f} - {estado}')
    print(f'  Estado: {estudiante.estado}')

# Resultados

In [ ]:
# 1. Generar datos aleatorios
print('\nGenera datos aleatorios de calificaciones para 50 estudiantes y 4 materias:')
estudiantes, materias = generar_datos_estudiantes()
print(f' • {len(estudiantes)} estudiantes creados')
print(f' • {len(materias)} materias configuradas')

In [ ]:
# 2. Calcular estadísticas
print('\nCalcula promedios y determina aprobación:')
estadisticas = calcular_estadisticas_generales(estudiantes, materias)
mostrar_ejemplo_estudiantes(estudiantes, 5)

In [ ]:
# 3. Generar visualizaciones
print('\nAnálisis Gráfico')
crear_graficas(estudiantes, materias)

In [ ]:
# 4. Resumen Estadístico
mostrar_resumen_estadistico(estadisticas)
